In [72]:
import pandas as pd
import json
import os
import sys
import gc
import re
import torch
import accelerate
import transformers
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download

In [73]:
current_path = os.getcwd()
print(f"📍 Current Directory: {current_path}")

📍 Current Directory: D:\Downloads\ipynb


In [74]:
# !pip uninstall torch torchvision torchaudio -y
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 --force-reinstall
# !pip install --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
# !pip install --no-cache-dir transformers==4.46.0 accelerate==0.34.0

In [75]:
def load_taxonomy(file_path="mtg_keywords.csv"):
    if not os.path.exists(file_path):
        print(f"⚠️ Error: {file_path} not found.")
        return []
        
    df = pd.read_csv(file_path).dropna(subset=['keyword', 'tag_name'])
    taxonomy = []
    
    for _, row in df.iterrows():
        raw_kw = str(row['keyword']).lower().strip()
        tag = str(row['tag_name']).lower().strip()
        
        # Safe Regex conversion
        pattern = re.escape(raw_kw).replace(r'\*', '.*?').replace(r'\?', '?')
        # Store as tuple to prevent unpacking errors
        taxonomy.append((pattern, tag))
        
    return taxonomy

In [80]:
HARD_TAXONOMY_MAP = load_taxonomy("mtg_keywords.csv")

if len(HARD_TAXONOMY_MAP) > 0:
    print(f"✅ Success! Loaded {len(HARD_TAXONOMY_MAP)} keyword pairs.")
    print(f"Example Pair: {HARD_TAXONOMY_MAP[0]}") 
else:
    print("❌ Failed to load taxonomy. Check if mtg_keywords.csv exists.")

✅ Success! Loaded 57 keyword pairs.
Example Pair: ('add\\ .*?\\ to\\ your\\ mana\\ pool\\.', 'mana-production')


In [81]:
# ==========================================
# CHECKPOINT 0: Environment & Version Check
# ==========================================
print(f"--- 🛠️ Environment Diagnostic ---")
print(f"Python:       {sys.version.split(' ')[0]}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Accelerate:   {accelerate.__version__}")

# GPU Verification
cuda_ready = torch.cuda.is_available()
print(f"CUDA Available: {cuda_ready}")
if cuda_ready:
    print(f"Device Name:    {torch.cuda.get_device_name(0)}")
    # Clear cache to start fresh on your 2080 Super
    torch.cuda.empty_cache()
else:
    print("⚠️ WARNING: GPU not detected. Script will be extremely slow.") 

# Force Python to find unreferenced objects and PyTorch to release VRAM
gc.collect()
torch.cuda.empty_cache()
print(f"🧹 GPU Cache Cleared. VRAM Reserved: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")

--- 🛠️ Environment Diagnostic ---
Python:       3.13.0
PyTorch:      2.6.0+cu124
Transformers: 4.46.0
Accelerate:   0.34.0
CUDA Available: True
Device Name:    NVIDIA GeForce RTX 2080 SUPER
🧹 GPU Cache Cleared. VRAM Reserved: 1332.00 MB


In [82]:
# ==========================================
# CHECKPOINT 1: Path & Model Config
# ==========================================
# ONLINE_ID = "valhalla/distilbart-mnli-12-3"
# LOCAL_PATH = os.path.abspath("./local_model/distilbart-mnli-12-3")
ONLINE_ID = "microsoft/Phi-3-mini-4k-instruct"
LOCAL_PATH = os.path.abspath("./local_model/phi3_model")

print(f"\n--- 📂 Model Path Check ---")
print(f"Local Path: {LOCAL_PATH}")

# Ensure local directory exists or download
if not os.path.exists(LOCAL_PATH):
    print("📥 Local model not found. Downloading...")
    snapshot_download(
        repo_id=ONLINE_ID,
        local_dir=LOCAL_PATH,
        local_dir_use_symlinks=False,
        allow_patterns=["*.json", "*.safetensors", "*.model", "*.py"]
    )
else:
    print("✅ Local model directory detected.")


--- 📂 Model Path Check ---
Local Path: D:\Downloads\ipynb\local_model\phi3_model
✅ Local model directory detected.


In [83]:
# ==========================================
# CHECKPOINT 2: Model & Tokenizer Loading
# ==========================================
print(f"\n--- 🧠 Loading Model to GPU ---")
try:
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_PATH, local_files_only=True)
    
    model = AutoModelForCausalLM.from_pretrained(
        LOCAL_PATH,
        torch_dtype=torch.float16,    # Optimized for 2080 Super VRAM
        device_map="auto",            # Automatic mapping to GPU
        trust_remote_code=True,
        local_files_only=True,
        attn_implementation="eager"    # Windows-friendly fallback
    )
    
    # Initialize Pipeline (Removal of 'device' arg is intentional for Accelerate)
    classifier = pipeline("text-generation", model=model, tokenizer=tokenizer)
    print(f"✅ Success! Pipeline active on: {next(model.parameters()).device}")

except Exception as e:
    print(f"❌ Initialization Failed: {e}")
    sys.exit("Execution stopped due to load error.")


--- 🧠 Loading Model to GPU ---


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


✅ Success! Pipeline active on: cuda:0


In [84]:
# ==========================================
# CHECKPOINT 3: Manual Feature Extraction
# ==========================================
def extract_manual_features(name, type_line, text):
    text_lower = text.lower() if text else ""
    
    # 1. Exact Mana Production
    # Handles: Dark Ritual -> {B}{B}{B}, Lands/Fixing -> {W}{B}
    if "dark ritual" in name.lower():
        mana_prod = "{B}{B}{B}"
    else:
        # Extract unique color symbols to identify production capability
        mana_symbols = re.findall(r'\{[WUBRG]\}|\{[WUBRG]/[WUBRG]\}', text)
        if mana_symbols:
            # Flatten hybrid symbols like {W/B} into individual colors W, B
            found_colors = set()
            for s in mana_symbols:
                for c in "WUBRG":
                    if c in s: found_colors.add(c)
            # Sort in WUBRG order for consistency
            order = "WUBRG"
            sorted_c = sorted(list(found_colors), key=lambda x: order.find(x))
            mana_prod = "".join([f"{{{c}}}" for c in sorted_c])
        elif "{C}" in text:
            mana_prod = "{C}"
        elif "add one mana of any color" in text_lower:
            mana_prod = "{Any}"
        else:
            mana_prod = "0"

    # 2. Card Draw (Numerical)
    draw_count = 0
    num_map = {"two": 2, "three": 3, "four": 4, "five": 5}
    draw_match = re.search(r"draw (\d+|two|three|four|five) card", text_lower)
    if draw_match:
        val = draw_match.group(1)
        draw_count = int(val) if val.isdigit() else num_map.get(val, 1)
    elif "draw a card" in text_lower:
        draw_count = 1

    # 3. Manual Tags (Keyword Taxonomy)
    manual_tags = []
    for pattern, tag_name in HARD_TAXONOMY_MAP:
        if re.search(pattern, text_lower, re.IGNORECASE):
            manual_tags.append(tag_name)

    return mana_prod, draw_count, manual_tags

In [85]:
# ==========================================
# CHECKPOINT 4: Hybrid Reasoning Loop
# ==========================================
VISIBLE_CP = 'mtg_classification_checkpoint.csv'
df = pd.read_csv('mtg_enriched_data.csv')
unique_cards = df[['name', 'type_line', 'effects_line']].drop_duplicates()
results = {}

# 1. Resume Logic (Same as before)
if os.path.exists(VISIBLE_CP):
    cp_df = pd.read_csv(VISIBLE_CP)
    for _, row in cp_df.drop_duplicates(subset=['name']).iterrows():
        def safe_list(v): 
            try: return ast.literal_eval(str(v)) if isinstance(v, str) else v
            except: return []
        results[row['name']] = {
            'mana_production': str(row.get('mana_production', '0')),
            'card_draw': int(row.get('card_draw', 0)),
            'manual_tags': safe_list(row.get('manual_tags')),
            'gen_tags': safe_list(row.get('generated_effect_tags'))
        }

# 2. Processing
cards_to_process = unique_cards[~unique_cards['name'].isin(results.keys())]

for i, (idx, row) in enumerate(cards_to_process.iterrows(), start=len(results)):
    print(f"   [{i+1}/{len(unique_cards)}] Analyzing: {row['name']}...")
    
    m_mana, m_draw, m_tags = extract_manual_features(row['name'], row['type_line'], row['effects_line'])
    
    # IMPROVED PROMPT: Clearer instructions for Phi-3
    tags_hint = ", ".join(m_tags) if m_tags else "None"
    prompt = (
        f"<|user|>\nIdentify 2 strategic MTG archetypes for this card.\n"
        f"Card Name: {row['name']}\n"
        f"Text: {row['effects_line']}\n"
        f"Keywords: {tags_hint}\n"
        f"Return the answer exactly in this format: {{\"gen_tags\": [\"tag1\", \"tag2\"]}}<|end|>\n"
        f"<|assistant|>"
    )
    
    g_tags = []
    try:
        # Generate more tokens to ensure it doesn't cut off the JSON
        res = classifier(prompt, max_new_tokens=60, do_sample=False, temperature=0.0)
        raw_output = res[0]['generated_text'].split("<|assistant|>")[-1].strip()
        
        # --- NEW REGEX FIX ---
        # Find anything between { and } in case the LLM adds chatter
        json_match = re.search(r"\{.*\}", raw_output, re.DOTALL)
        if json_match:
            clean_json = json_match.group(0).replace("'", '"')
            llm_data = json.loads(clean_json)
            g_tags = llm_data.get('gen_tags', [])
        else:
            print(f"      ⚠️ No JSON found in output for {row['name']}")
    except Exception as e:
        print(f"      ❌ Parsing Error for {row['name']}: {e}")
        g_tags = []

    results[row['name']] = {
        'mana_production': m_mana, 'card_draw': m_draw, 'manual_tags': m_tags, 'gen_tags': g_tags
    }

    # Batch Save (Every 10)
    if (i + 1) % 10 == 0:
        checkpoint_df = pd.DataFrame.from_dict(results, orient='index').reset_index()
        checkpoint_df.columns = ['name', 'mana_production', 'card_draw', 'manual_tags', 'generated_effect_tags']
        checkpoint_df.to_csv(VISIBLE_CP, index=False)
        torch.cuda.empty_cache()

# Final Export
df['mana_production'] = df['name'].map(lambda x: results[x]['mana_production'])
df['card_draw'] = df['name'].map(lambda x: results[x]['card_draw'])
df['manual_tags'] = df['name'].map(lambda x: results[x]['manual_tags'])
df['generated_effect_tags'] = df['name'].map(lambda x: results[x]['gen_tags'])
df.to_csv('mtg_final_classified_deck.csv', index=False)

C:\Users\Charlie\anaconda3\Lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


   [1/95] Analyzing: Adeline, Resplendent Cathar...
   [2/95] Analyzing: Adriana, Captain of the Guard...
   [3/95] Analyzing: Agadeem's Awakening // Agadeem, the Undercrypt...
   [4/95] Analyzing: Akroma's Will...
   [5/95] Analyzing: Anthem of Rakdos...
   [6/95] Analyzing: Arcane Signet...
   [7/95] Analyzing: Archon of Emeria...
   [8/95] Analyzing: Aurelia, the Warleader...
   [9/95] Analyzing: Aven Mindcensor...
   [10/95] Analyzing: Battlefield Forge...
   [11/95] Analyzing: Blood Crypt...


C:\Users\Charlie\anaconda3\Lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


   [12/95] Analyzing: Boros Signet...
   [13/95] Analyzing: Breena, the Demagogue...
   [14/95] Analyzing: Brimaz, King of Oreskos...
   [15/95] Analyzing: Captain Lannery Storm...
   [16/95] Analyzing: Castle Embereth...
   [17/95] Analyzing: Caves of Koilos...
   [18/95] Analyzing: City of Brass...
   [19/95] Analyzing: Combat Calligrapher...
   [20/95] Analyzing: Combat Celebrant...
   [21/95] Analyzing: Command Tower...


C:\Users\Charlie\anaconda3\Lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


...

   [72/95] Analyzing: Sacred Foundry...
   [73/95] Analyzing: Savai Triome...
   [74/95] Analyzing: Serra Ascendant...
   [75/95] Analyzing: Shadowblood Ridge...
   [76/95] Analyzing: Olivia, Crimson Bride...
   [77/95] Analyzing: Skullclamp...
   [78/95] Analyzing: Skyknight Vanguard...
   [79/95] Analyzing: Smothering Tithe...
   [80/95] Analyzing: Sol Ring...
   [81/95] Analyzing: Spectator Seating...


C:\Users\Charlie\anaconda3\Lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


   [82/95] Analyzing: Swamp...
   [83/95] Analyzing: Swiftfoot Boots...
   [84/95] Analyzing: Takenuma, Abandoned Mire...
   [85/95] Analyzing: Talisman of Conviction...
   [86/95] Analyzing: Talisman of Hierarchy...
   [87/95] Analyzing: Thalia, Guardian of Thraben...
   [88/95] Analyzing: The Restoration of Eiganjo // Architect of Restoration...
   [89/95] Analyzing: Tithe...
   [90/95] Analyzing: Tocatli Honor Guard...
   [91/95] Analyzing: Triumphant Adventurer...


C:\Users\Charlie\anaconda3\Lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


   [92/95] Analyzing: Vault of Champions...
   [93/95] Analyzing: Waves of Aggression...
   [94/95] Analyzing: Winota, Joiner of Forces...
   [95/95] Analyzing: Isshin, Two Heavens as One...


In [86]:
# # --- 🛑 THE KILL SWITCH ---
# # Remove this block once you have your GPU environment set up correctly
# if not next(model.parameters()).is_cuda:
#     print("\n🛑 STOPPING RUN: Model is on CPU.")
#     print("Classification on CPU will be extremely slow for a full deck.")
#     print("Please request a GPU (srun/sbatch) or run this on your 2080 Super.")
#     sys.exit("Execution halted to save time.")

In [87]:
# ==========================================
# FINAL STEP: Hide Checkpoint (Archival Only)
# ==========================================
VISIBLE_CP = 'mtg_classification_checkpoint.csv'
HIDDEN_CP = '.mtg_classification_checkpoint.csv'

if os.path.exists(VISIBLE_CP):
    # Remove previous hidden archive if it exists
    if os.path.exists(HIDDEN_CP):
        os.remove(HIDDEN_CP)
    
    os.rename(VISIBLE_CP, HIDDEN_CP)
    print(f"✅ Run complete. Checkpoint archived as hidden file: {HIDDEN_CP}")
else:
    print("ℹ️ No visible checkpoint found to rename.")

✅ Run complete. Checkpoint archived as hidden file: .mtg_classification_checkpoint.csv
